# Stress Test GPU 0 + GPU 1 + CPU

Notebook ini membebani kedua GPU dan CPU **bersamaan**, menggunakan perkalian matriks PyTorch dalam proses terpisah. File disiapkan tanpa menjalankan sel apa pun.

**Prasyarat:** Linux, kernel Python 3.10+, PyTorch dengan dukungan CUDA, `psutil`, serta `nvidia-smi` yang bisa membaca suhu GPU. Pilih environment yang sudah memiliki paket tersebut; notebook tidak menginstal paket atau mengubah driver, clock, power limit, layanan, maupun konfigurasi sistem.

**Server bersama:** jalankan hanya dengan izin pengelola dan saat tidak mengganggu pekerjaan lain. Default: 60 detik, 4 pekerja CPU, matriks GPU FP16 berukuran 8192, serta penghentian pada suhu GPU 85 C. Ambang ini dapat diubah sesuai spesifikasi dan kebijakan perangkat, bukan jaminan keamanan perangkat keras. Stress test bisa menaikkan suhu/daya dan memperlambat layanan lain.

## 1. Impor dan Konfigurasi

- `GPU_IDS=[0, 1]` adalah indeks **CUDA yang terlihat oleh kernel**. `CUDA_VISIBLE_DEVICES` atau container dapat mengubah pemetaannya ke GPU fisik. Pemetaan UUID/indeks fisik akan ditampilkan sebelum konfirmasi. Notebook tidak membuka GPU yang disembunyikan administrator dan menolak berjalan bila kedua GPU tidak tersedia.
- `CPU_WORKERS=4` berarti 4 proses, masing-masing 1 thread komputasi. Naikkan hanya sesuai alokasi CPU Anda; kuota container tetap berlaku. Sisakan kapasitas untuk sistem dan proses GPU.
- `GPU_MAX_FREE_MEMORY_FRACTION=0.25` membatasi estimasi alokasi terhadap VRAM kosong, bukan target mengisi VRAM. Tiga matriks default memakai sekitar 384 MiB per GPU, di luar runtime CUDA dan ruang kerja library.
- Pengujian baru dapat dimulai setelah `CONFIRM_STRESS_TEST=True` dan konfirmasi teks pada sel eksekusi. `Run All` dengan konfigurasi awal tidak memulai beban.
- Durasi mencakup startup proses. Penghentian/pembersihan dapat menambah beberapa detik. Ini uji beban komputasi, bukan tes menyeluruh integritas RAM/VRAM atau sertifikasi kestabilan.


In [ ]:
import csv
import json
import math
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import UUID

import psutil
import torch

GPU_IDS = [0, 1]
DURATION_SECONDS = 60
CPU_WORKERS = 4
GPU_MATRIX_SIZE = 8192
CPU_MATRIX_SIZE = 1024
GPU_MAX_FREE_MEMORY_FRACTION = 0.25
MONITOR_INTERVAL_SECONDS = 2
MAX_GPU_TEMP_C = 85
MAX_RAM_PERCENT = 90
CONFIRM_STRESS_TEST = False


def normalize_uuid(value):
    if isinstance(value, bytes):
        value = str(UUID(bytes=value)) if len(value) == 16 else value.decode('ascii')
    return str(value).strip().lower().removeprefix('gpu-')


def numeric_or_none(value):
    try:
        number = float(value.strip())
        return number if math.isfinite(number) else None
    except ValueError:
        return None


def gpu_metrics():
    fields = 'index,uuid,name,temperature.gpu,utilization.gpu,memory.used,memory.total,power.draw'
    report = subprocess.run(
        ['nvidia-smi', f'--query-gpu={fields}', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=True, timeout=5,
    )
    metrics = {}
    for row in csv.reader(report.stdout.splitlines(), skipinitialspace=True):
        if len(row) != 8:
            raise RuntimeError('Format keluaran nvidia-smi tidak dikenali.')
        metrics[normalize_uuid(row[1])] = {
            'physical_gpu_id': int(row[0]),
            'gpu_uuid': row[1].strip(),
            'gpu_name': row[2].strip(),
            'gpu_temperature_c': numeric_or_none(row[3]),
            'gpu_util_percent': numeric_or_none(row[4]),
            'gpu_memory_used_mib': numeric_or_none(row[5]),
            'gpu_memory_total_mib': numeric_or_none(row[6]),
            'gpu_power_w': numeric_or_none(row[7]),
        }
    return metrics


def check_limits(mapping, metrics):
    for gpu_id, device_uuid in mapping.items():
        if device_uuid not in metrics:
            raise RuntimeError(f'Telemetri GPU CUDA {gpu_id} hilang.')
        temperature = metrics[device_uuid]['gpu_temperature_c']
        if temperature is None:
            raise RuntimeError(f'Sensor suhu GPU CUDA {gpu_id} tidak tersedia.')
        if temperature >= MAX_GPU_TEMP_C:
            raise RuntimeError(f'GPU CUDA {gpu_id}: suhu {temperature} C mencapai batas.')
    if psutil.virtual_memory().percent >= MAX_RAM_PERCENT:
        raise RuntimeError('Pemakaian RAM sistem mencapai batas penghentian.')


def preflight():
    if not 0 < DURATION_SECONDS <= 3600:
        raise ValueError('DURATION_SECONDS harus lebih dari 0 dan maksimal 3600.')
    if not 0 < MONITOR_INTERVAL_SECONDS <= 10:
        raise ValueError('Interval pemantauan harus lebih dari 0 dan maksimal 10 detik.')
    if not 0 < GPU_MAX_FREE_MEMORY_FRACTION <= 0.5:
        raise ValueError('Fraksi VRAM kosong harus lebih dari 0 dan maksimal 0.5.')
    if not 40 <= MAX_GPU_TEMP_C <= 90 or not 10 <= MAX_RAM_PERCENT <= 95:
        raise ValueError('Batas suhu GPU harus 40-90 C; batas RAM harus 10-95 persen.')
    available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
    if type(CPU_WORKERS) is not int or not 1 <= CPU_WORKERS <= available_cpus:
        raise ValueError(f'CPU_WORKERS harus 1 sampai {available_cpus}, sesuai alokasi Anda.')
    for label, size, maximum in [('GPU', GPU_MATRIX_SIZE, 16384), ('CPU', CPU_MATRIX_SIZE, 4096)]:
        if type(size) is not int or not 256 <= size <= maximum or size % 256:
            raise ValueError(f'Ukuran matriks {label} harus kelipatan 256, antara 256 dan {maximum}.')
    if not isinstance(GPU_IDS, (list, tuple)) or len(GPU_IDS) != 2:
        raise ValueError('Pilih tepat dua GPU melalui GPU_IDS, default [0, 1].')
    if any(type(gpu_id) is not int or gpu_id < 0 for gpu_id in GPU_IDS) or len(set(GPU_IDS)) != 2:
        raise ValueError('GPU_IDS harus dua indeks integer berbeda dan tidak negatif.')
    if not shutil.which('nvidia-smi'):
        raise RuntimeError('nvidia-smi diperlukan untuk pemantauan suhu; pengujian dibatalkan.')
    if not torch.cuda.is_available() or max(GPU_IDS) >= torch.cuda.device_count():
        raise RuntimeError('Kedua GPU yang diminta tidak terlihat oleh CUDA. Tidak ada fallback CPU.')

    metrics = gpu_metrics()
    mapping = {}
    required_gpu_bytes = 3 * GPU_MATRIX_SIZE ** 2 * 2 + 512 * 1024 ** 2
    for gpu_id in GPU_IDS:
        properties = torch.cuda.get_device_properties(gpu_id)
        device_uuid = normalize_uuid(getattr(properties, 'uuid', None))
        if device_uuid not in metrics:
            raise RuntimeError('UUID PyTorch tidak cocok dengan nvidia-smi; pemetaan GPU tidak dapat diverifikasi.')
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
        if required_gpu_bytes > free_bytes * GPU_MAX_FREE_MEMORY_FRACTION:
            raise RuntimeError(f'VRAM kosong GPU {gpu_id} tidak cukup; kurangi GPU_MATRIX_SIZE.')
        mapping[gpu_id] = device_uuid
        physical_id = metrics[device_uuid]['physical_gpu_id']
        print(f'CUDA {gpu_id} -> GPU fisik {physical_id}: {properties.name}; UUID {device_uuid}')
        print(f'  VRAM kosong/total: {free_bytes / 1024 ** 3:.2f}/{total_bytes / 1024 ** 3:.2f} GiB')
    required_ram = CPU_WORKERS * (3 * CPU_MATRIX_SIZE ** 2 * 4 + 512 * 1024 ** 2)
    required_ram += len(GPU_IDS) * 512 * 1024 ** 2
    if required_ram > psutil.virtual_memory().available * 0.5:
        raise RuntimeError('Estimasi RAM terlalu besar; kurangi CPU_WORKERS atau ukuran matriks.')
    check_limits(mapping, metrics)
    print(f'Python: {sys.executable}; PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}')
    print(f'CUDA_VISIBLE_DEVICES: {os.environ.get("CUDA_VISIBLE_DEVICES", "tidak disetel")}')
    print(f'CPU affinity: {available_cpus}; pekerja CPU: {CPU_WORKERS}; durasi: {DURATION_SECONDS} detik')
    print('Kuota CPU/RAM container tetap berlaku. Suhu CPU tidak dipantau oleh notebook ini.')
    return mapping


WORKER_CODE = r'''
import json
import os
import sys
import time

parameters = json.loads(sys.argv[1])
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)
parent_pid = parameters['parent_pid']
deadline = parameters['deadline']
if os.getppid() != parent_pid or time.monotonic() >= deadline:
    raise SystemExit('Induk sudah berhenti atau durasi habis saat startup.')

use_gpu = parameters['kind'] == 'gpu'
device = torch.device('cuda:' + str(parameters['gpu_id']) if use_gpu else 'cpu')
dtype = torch.float16 if use_gpu else torch.float32
size = parameters['matrix_size']
if use_gpu:
    torch.cuda.set_device(device)
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    required_bytes = 3 * size ** 2 * 2 + 512 * 1024 ** 2
    if required_bytes > free_bytes * parameters['memory_fraction']:
        raise RuntimeError('VRAM kosong berubah; alokasi GPU dibatalkan.')

iterations = 0
with torch.no_grad():
    left = torch.randn((size, size), dtype=dtype, device=device)
    right = torch.randn((size, size), dtype=dtype, device=device)
    result = torch.empty_like(left)
    while os.getppid() == parent_pid and time.monotonic() < deadline:
        torch.mm(left, right, out=result)
        if use_gpu:
            torch.cuda.synchronize(device)
        iterations += 1
print(json.dumps({'device': str(device), 'iterations': iterations}), flush=True)
'''


def stop_workers(workers):
    forced = set()
    for label, process in workers:
        if process.poll() is None:
            forced.add(label)
            process.terminate()
    cleanup_deadline = time.monotonic() + 5
    summaries = []
    for label, process in workers:
        try:
            output, _ = process.communicate(timeout=max(0.1, cleanup_deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            process.kill()
            output, _ = process.communicate(timeout=5)
        summaries.append({
            'worker': label, 'pid': process.pid, 'returncode': process.returncode,
            'stopped_by_notebook': label in forced, 'output': output.strip(),
        })
    return summaries


def run_stress_test():
    if CONFIRM_STRESS_TEST is not True:
        raise RuntimeError('Aktifkan CONFIRM_STRESS_TEST secara manual sebelum memulai.')
    mapping = preflight()
    confirmation = input('Ketik MULAI STRESS untuk menjalankan kedua GPU dan CPU: ').strip()
    if confirmation != 'MULAI STRESS':
        print('Dibatalkan. Tidak ada pekerja stress yang dimulai.')
        return {'status': 'dibatalkan', 'metrics': [], 'workers': []}

    worker_environment = os.environ.copy()
    for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        worker_environment[variable] = '1'
    specifications = [
        (f'GPU-{gpu_id}', 'gpu', gpu_id, GPU_MATRIX_SIZE) for gpu_id in GPU_IDS
    ] + [
        (f'CPU-{worker_id}', 'cpu', None, CPU_MATRIX_SIZE) for worker_id in range(CPU_WORKERS)
    ]
    workers = []
    history = []
    status = 'durasi selesai'
    started = time.monotonic()
    deadline = started + DURATION_SECONDS
    psutil.cpu_percent(interval=None)
    try:
        for label, kind, gpu_id, matrix_size in specifications:
            if time.monotonic() >= deadline:
                raise RuntimeError('Durasi habis sebelum semua pekerja sempat dimulai.')
            parameters = {
                'kind': kind, 'gpu_id': gpu_id, 'matrix_size': matrix_size,
                'deadline': deadline, 'parent_pid': os.getpid(),
                'memory_fraction': GPU_MAX_FREE_MEMORY_FRACTION,
            }
            environment = worker_environment.copy()
            if kind == 'cpu':
                environment['CUDA_VISIBLE_DEVICES'] = ''
            process = subprocess.Popen(
                [sys.executable, '-u', '-c', WORKER_CODE, json.dumps(parameters)],
                env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, start_new_session=True,
            )
            workers.append((label, process))
            print(f'{label} dimulai, PID {process.pid}', flush=True)
        while time.monotonic() < deadline:
            for label, process in workers:
                returncode = process.poll()
                if returncode is not None and (returncode != 0 or time.monotonic() < deadline):
                    raise RuntimeError(f'{label} berhenti lebih awal, exit={returncode}. Lihat ringkasan worker.')
            metrics = gpu_metrics()
            check_limits(mapping, metrics)
            elapsed = time.monotonic() - started
            cpu_percent = psutil.cpu_percent(interval=None)
            ram_percent = psutil.virtual_memory().percent
            timestamp = datetime.now(timezone.utc).isoformat()
            gpu_text = []
            for gpu_id, device_uuid in mapping.items():
                row = {
                    'timestamp_utc': timestamp, 'elapsed_seconds': round(elapsed, 2),
                    'cuda_gpu_id': gpu_id, 'cpu_percent': cpu_percent,
                    'ram_percent': ram_percent, **metrics[device_uuid],
                }
                history.append(row)
                gpu_text.append(
                    f"GPU {gpu_id}: {row['gpu_util_percent']}%, {row['gpu_temperature_c']} C, "
                    f"{row['gpu_memory_used_mib']} MiB, {row['gpu_power_w']} W"
                )
            print(f'{elapsed:5.1f}s | CPU sistem {cpu_percent}% | RAM {ram_percent}% | ' + ' | '.join(gpu_text), flush=True)
            time.sleep(min(MONITOR_INTERVAL_SECONDS, max(0, deadline - time.monotonic())))
    except KeyboardInterrupt:
        status = 'dihentikan pengguna'
        print('Interrupt diterima; menghentikan pekerja milik notebook.', flush=True)
    except Exception:
        status = 'gagal atau batas keselamatan tercapai'
        raise
    finally:
        summaries = stop_workers(workers)
        print(f'Pengujian berakhir: {status}.', flush=True)
        for summary in summaries:
            print(f"{summary['worker']} (PID {summary['pid']}), exit={summary['returncode']}")
            if summary['output']:
                print(summary['output'][-2000:])
    failed = [item['worker'] for item in summaries if item['returncode'] != 0 and not item['stopped_by_notebook']]
    if failed and status == 'durasi selesai':
        raise RuntimeError(f'Pekerja gagal: {failed}. Pengujian tidak dinyatakan berhasil.')
    return {'status': status, 'metrics': history, 'workers': summaries}


## 2. Jalankan Beban Paralel Secara Manual

Sel kode sebelumnya hanya berisi impor, konfigurasi, dan definisi fungsi. `preflight()` memeriksa kedua GPU, mencocokkan UUID dengan `nvidia-smi`, memperkirakan kebutuhan memori, dan memeriksa suhu sebelum konfirmasi. `WORKER_CODE` disimpan sebagai teks di dalam notebook, sehingga tidak membutuhkan file Python pendamping.

Setiap GPU memperoleh proses Python sendiri untuk perkalian matriks FP16. CPU memakai proses terpisah dengan matriks FP32 dan satu thread per proses, sehingga tidak terhambat GIL Python atau menggandakan thread BLAS tanpa batas. Kesalahan CUDA/OOM pada satu pekerja menghentikan pengujian lainnya.

Untuk menjalankan **nanti**, setelah mendapat izin:

1. Pilih kernel dengan PyTorch CUDA dan `psutil` yang sudah terpasang.
2. Sesuaikan konfigurasi, ubah `CONFIRM_STRESS_TEST=True`, lalu jalankan sel definisi di atas.
3. Jalankan sel di bawah, periksa pemetaan GPU yang tercetak, kemudian ketik `MULAI STRESS` bila sesuai.

Utilisasi CPU/RAM yang ditampilkan adalah **tingkat sistem**, bukan hanya proses pengujian. Telemetri GPU juga mencakup pekerjaan lain pada GPU yang sama. Nilai `None` berarti metrik tidak tersedia, bukan nol; suhu GPU yang tidak terbaca akan menghentikan pengujian. Utilisasi 100% tidak dijamin karena bergantung pada perangkat, ukuran matriks, kuota, dan beban lain.


In [ ]:
stress_result = None

if CONFIRM_STRESS_TEST is True:
    stress_result = run_stress_test()
else:
    print('Stress test TIDAK dimulai. CONFIRM_STRESS_TEST masih False.')


## 3. Hentikan dan Tinjau Hasil

- Tekan **Interrupt / Stop Execution** pada notebook untuk menghentikan lebih awal. Sel penghenti terpisah tidak akan membantu saat kernel sedang sibuk karena eksekusinya mengantre.
- Blok `finally` mengirim penghentian hanya ke proses yang dibuat notebook, menunggu proses keluar, lalu memakai `kill()` pada proses tersebut saja jika belum selesai. Tidak ada `pkill`, penghentian layanan, atau manipulasi proses pengguna lain.
- Setiap pekerja juga memiliki tenggat waktu dan memeriksa apakah kernel induknya masih hidup di antara operasi matriks. Menutup tab saja **tidak** menghentikan kernel. Operasi native/driver yang macet dapat membutuhkan intervensi pengelola; watchdog ini bukan perlindungan perangkat keras.
- Suhu GPU diperiksa secara berkala, bukan seketika. Suhu CPU tidak dipantau: gunakan sensor dan kebijakan pengelola untuk CPU. Jika pembacaan GPU gagal atau batas suhu/RAM terlewati, seluruh pekerja dihentikan dan sel menampilkan galat.
- Setelah selesai, alokasi matriks besar dilepas saat proses pekerja keluar. Konteks CUDA kecil milik kernel untuk pemeriksaan perangkat bisa tetap ada sampai kernel di-restart.
- `stress_result` berisi status, metrik, dan ringkasan proses bila selesai normal atau diinterupsi. Jika ada galat, lihat traceback dan ringkasan worker yang tercetak; hasil bukan bukti bahwa perangkat lolos uji kestabilan.

## 4. Ekspor CSV dan Grafik Opsional

Sel berikut tidak menambahkan beban stress. Ekspor dan grafik **nonaktif secara default**. Setelah memiliki hasil, ubah `SAVE_CSV` atau `PLOT_RESULTS` menjadi `True`. CSV dibuat dengan nama waktu yang unik di direktori kerja notebook, tanpa menimpa file lama. Grafik membutuhkan `matplotlib` yang sudah tersedia; tidak ada instalasi otomatis.


In [ ]:
SAVE_CSV = False
PLOT_RESULTS = False

if stress_result and stress_result['metrics']:
    samples = stress_result['metrics']
    print(f"Status: {stress_result['status']}; jumlah baris metrik: {len(samples)}")
    if SAVE_CSV:
        timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
        csv_path = Path.cwd() / f'stress_metrics_{timestamp}.csv'
        with csv_path.open('x', newline='', encoding='utf-8') as output_file:
            writer = csv.DictWriter(output_file, fieldnames=list(samples[0]))
            writer.writeheader()
            writer.writerows(samples)
        print(f'CSV: {csv_path}')
    if PLOT_RESULTS:
        try:
            import matplotlib.pyplot as plt
        except ImportError:
            print('Grafik dilewati: matplotlib belum tersedia pada kernel ini.')
        else:
            figure, axes = plt.subplots(1, 2, figsize=(12, 4))
            measured_gpu_ids = sorted({sample['cuda_gpu_id'] for sample in samples})
            for gpu_id in measured_gpu_ids:
                selected = [sample for sample in samples if sample['cuda_gpu_id'] == gpu_id]
                seconds = [sample['elapsed_seconds'] for sample in selected]
                axes[0].plot(seconds, [sample['gpu_util_percent'] for sample in selected], label=f'GPU {gpu_id}')
                axes[1].plot(seconds, [sample['gpu_temperature_c'] for sample in selected], label=f'GPU {gpu_id}')
            cpu_samples = [sample for sample in samples if sample['cuda_gpu_id'] == measured_gpu_ids[0]]
            axes[0].plot(
                [sample['elapsed_seconds'] for sample in cpu_samples],
                [sample['cpu_percent'] for sample in cpu_samples],
                label='CPU sistem', linestyle='--',
            )
            axes[0].set_ylabel('Utilisasi (%)')
            axes[0].set_ylim(0, 105)
            axes[1].set_ylabel('Suhu GPU (C)')
            axes[1].axhline(MAX_GPU_TEMP_C, color='red', linestyle=':', label='Batas suhu')
            for axis in axes:
                axis.set_xlabel('Waktu (detik)')
                axis.grid(alpha=0.3)
                axis.legend()
            figure.tight_layout()
            plt.show()
else:
    print('Belum ada hasil stress test. Tidak ada CSV atau grafik yang dibuat.')
